# Deep Learning for Joint OD Word-Bit Prediction (8-bit ChaCha)

We build a multi-label target over 16 words x 8 bits = 128 outputs.
Each output target y[w*8 + b] corresponds to whether bit b of CipherTextDiff_w is 1.
The model learns P(OD_word=w AND OD_bit=b | (ID_word, ID_bit, plaintext context)).

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import joblib
import warnings
# warnings.filterwarnings('ignore')

## 1) Load Dataset and Prepare Features

In [3]:
# Load the dataset
dataset = pd.read_csv('dataset.csv')
print(f"Dataset shape: {dataset.shape}")
print(f"Columns: {dataset.columns.tolist()}")
print(f"First few rows:")
print(dataset.head())

Dataset shape: (320000, 34)
Columns: ['IDWordIndex', 'IDBitIndex', 'PlainText_0', 'PlainText_1', 'PlainText_2', 'PlainText_3', 'PlainText_4', 'PlainText_5', 'PlainText_6', 'PlainText_7', 'PlainText_8', 'PlainText_9', 'PlainText_10', 'PlainText_11', 'PlainText_12', 'PlainText_13', 'PlainText_14', 'PlainText_15', 'CipherTextDiff_0', 'CipherTextDiff_1', 'CipherTextDiff_2', 'CipherTextDiff_3', 'CipherTextDiff_4', 'CipherTextDiff_5', 'CipherTextDiff_6', 'CipherTextDiff_7', 'CipherTextDiff_8', 'CipherTextDiff_9', 'CipherTextDiff_10', 'CipherTextDiff_11', 'CipherTextDiff_12', 'CipherTextDiff_13', 'CipherTextDiff_14', 'CipherTextDiff_15']
First few rows:
  IDWordIndex  IDBitIndex  PlainText_0 PlainText_1  PlainText_2  PlainText_3  \
0           c           0           65          6e           32           74   
1           c           1           65          6e           32           74   
2           c           2           65          6e           32           74   
3           c           3

In [4]:
# Encode input word symbol (e.g., 'c','d','e','f') to integer id
le_word_joint = LabelEncoder()
unique_words = dataset['IDWordIndex'].astype(str)
le_word_joint.fit(unique_words)
print(f"Unique input words: {le_word_joint.classes_}")

# Utility: hex string (8-bit) -> 8-bit binary array
def hex_to_bits8(x: str) -> np.ndarray:
    """Convert hex string to 8-bit binary array"""
    # Handle single hex digits by padding
    hex_str = str(x).strip()
    if len(hex_str) == 1:
        hex_str = '0' + hex_str
    
    # Convert hex to int, then to 8-bit binary string
    try:
        val = int(hex_str, 16)
        s = format(val, '08b')
        return np.array([1 if ch=='1' else 0 for ch in s], dtype=np.uint8)
    except ValueError:
        # If conversion fails, return all zeros
        return np.zeros(8, dtype=np.uint8)

# Test the function
print(f"Test hex_to_bits8('c1'): {hex_to_bits8('c1')}")
print(f"Test hex_to_bits8('7'): {hex_to_bits8('7')}")

Unique input words: ['c' 'd' 'e' 'f']
Test hex_to_bits8('c1'): [1 1 0 0 0 0 0 1]
Test hex_to_bits8('7'): [0 0 0 0 0 1 1 1]


In [5]:
# Build X features: [input_word_id, input_bit]
# Build Y labels: 128-dimensional binary vector for all output word-bit combinations
X_list = []
Y_list = []

print("Processing dataset...")
# Iterate rows once; for each row create a 128-dim label
for idx, row in dataset.iterrows():
    if idx % 10000 == 0:
        print(f"Processed {idx} rows")
    
    input_word_id = le_word_joint.transform([str(row['IDWordIndex'])])[0]
    input_bit = int(row['IDBitIndex'])

    # Feature vector: [input_word_id, input_bit]
    X_list.append([input_word_id, input_bit])

    # Build 128 labels by concatenating bits of each CipherTextDiff_w (w=0..15)
    bits_all = []
    for w in range(16):
        bits = hex_to_bits8(row[f'CipherTextDiff_{w}'])  # shape (8,)
        bits_all.append(bits)
    y = np.concatenate(bits_all, axis=0)  # shape (128,)
    Y_list.append(y)

X_np = np.asarray(X_list, dtype=np.int64)            # shape (N, 2)
Y_np = np.asarray(Y_list, dtype=np.uint8)           # shape (N, 128)

print(f"\nFeature matrix shape: {X_np.shape}")
print(f"Label matrix shape: {Y_np.shape}")
print(f"Average number of active bits per sample: {Y_np.sum(axis=1).mean():.2f}")
print(f"Percentage of active bits: {(Y_np.sum() / Y_np.size * 100):.2f}%")

Processing dataset...
Processed 0 rows
Processed 10000 rows
Processed 20000 rows
Processed 30000 rows
Processed 40000 rows
Processed 50000 rows
Processed 60000 rows
Processed 70000 rows
Processed 80000 rows
Processed 90000 rows
Processed 100000 rows
Processed 110000 rows
Processed 120000 rows
Processed 130000 rows
Processed 140000 rows
Processed 150000 rows
Processed 160000 rows
Processed 170000 rows
Processed 180000 rows
Processed 190000 rows
Processed 200000 rows
Processed 210000 rows
Processed 220000 rows
Processed 230000 rows
Processed 240000 rows
Processed 250000 rows
Processed 260000 rows
Processed 270000 rows
Processed 280000 rows
Processed 290000 rows
Processed 300000 rows
Processed 310000 rows

Feature matrix shape: (320000, 2)
Label matrix shape: (320000, 128)
Average number of active bits per sample: 63.19
Percentage of active bits: 49.37%


In [6]:
# Standardize numeric features for DL stability
scaler_joint = StandardScaler()
X_np_scaled = scaler_joint.fit_transform(X_np.astype(np.float32))

# Train/val split
# Use stratification based on whether any output bits are active
stratify_labels = (Y_np.sum(axis=1) > 0).astype(int)
X_train, X_val, y_train, y_val = train_test_split(
    X_np_scaled, Y_np, test_size=0.2, random_state=42, stratify=stratify_labels
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

Training set: 256000 samples
Validation set: 64000 samples


## 2) PyTorch Dataset and DataLoader

In [7]:
class ODJointDataset(Dataset):
    def __init__(self, X: np.ndarray, Y: np.ndarray):
        self.X = torch.from_numpy(X).float()
        self.Y = torch.from_numpy(Y).float()
    
    def __len__(self):
        return self.X.shape[0]
    
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

batch_size = 1024
train_ds = ODJointDataset(X_train, y_train)
val_ds = ODJointDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

Training batches: 250
Validation batches: 63


## 3) Define MLP Model for 128 Outputs

In [8]:
class ODJointMLP(nn.Module):
    def __init__(self, in_dim: int = 2, hidden: int = 256, out_dim: int = 128, p_drop: float = 0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(p_drop),
            nn.Linear(hidden, out_dim)
        )
    
    def forward(self, x):
        return self.net(x)  # logits for 128 labels

# Instantiate model, loss (BCE-with-logits), optimizer
model_joint = ODJointMLP(in_dim=X_train.shape[1], hidden=256, out_dim=128, p_drop=0.2)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model_joint.parameters(), lr=1e-3)

print(f"Model parameters: {sum(p.numel() for p in model_joint.parameters()):,}")
print(f"Model output dimension: 128 (16 words × 8 bits)")

Model parameters: 99,456
Model output dimension: 128 (16 words × 8 bits)


## 4) Training Loop

In [9]:
def evaluate(model: nn.Module, data_loader: DataLoader):
    model.eval()
    total_loss = 0.0
    y_true_all = []
    y_prob_all = []
    
    with torch.no_grad():
        for xb, yb in data_loader:
            logits = model(xb)
            loss = criterion(logits, yb)
            total_loss += loss.item() * xb.size(0)
            
            probs = torch.sigmoid(logits).cpu().numpy()
            y_prob_all.append(probs)
            y_true_all.append(yb.cpu().numpy())
    
    y_true = np.concatenate(y_true_all, axis=0)
    y_prob = np.concatenate(y_prob_all, axis=0)
    
    # Macro AUPRC/AUROC across 128 outputs
    try:
        # Only compute metrics for labels that have positive examples
        valid_labels = y_true.sum(axis=0) > 0
        if valid_labels.sum() > 0:
            ap = average_precision_score(y_true[:, valid_labels], y_prob[:, valid_labels], average='macro')
            auroc = roc_auc_score(y_true[:, valid_labels], y_prob[:, valid_labels], average='macro')
        else:
            ap = auroc = np.nan
    except Exception:
        ap = auroc = np.nan
    
    avg_loss = total_loss / len(data_loader.dataset)
    return avg_loss, ap, auroc

In [10]:
epochs = 10  # adjust upward if needed
best_val_ap = -1.0
best_state_dict = None

for epoch in range(1, epochs+1):
    model_joint.train()
    running = 0.0
    
    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model_joint(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running += loss.item() * xb.size(0)
    
    train_loss = running / len(train_loader.dataset)
    val_loss, val_ap, val_auc = evaluate(model_joint, val_loader)
    
    print(f"Epoch {epoch:02d} | train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_mAP={val_ap:.4f} val_AUROC={val_auc:.4f}")
    
    # Track best by validation mAP
    if (not np.isnan(val_ap)) and val_ap > best_val_ap:
        best_val_ap = val_ap
        best_state_dict = {k: v.cpu().clone() for k, v in model_joint.state_dict().items()}

# Load best weights if improved
if best_state_dict is not None:
    model_joint.load_state_dict(best_state_dict)
    print(f"\nLoaded best model with validation mAP: {best_val_ap:.4f}")

Epoch 01 | train_loss=0.6791 val_loss=0.6677 val_mAP=0.5913 val_AUROC=0.6035
Epoch 02 | train_loss=0.6647 val_loss=0.6566 val_mAP=0.6110 val_AUROC=0.6169
Epoch 03 | train_loss=0.6589 val_loss=0.6544 val_mAP=0.6132 val_AUROC=0.6188
Epoch 04 | train_loss=0.6572 val_loss=0.6537 val_mAP=0.6134 val_AUROC=0.6190
Epoch 05 | train_loss=0.6564 val_loss=0.6536 val_mAP=0.6134 val_AUROC=0.6190
Epoch 06 | train_loss=0.6559 val_loss=0.6534 val_mAP=0.6134 val_AUROC=0.6189
Epoch 07 | train_loss=0.6557 val_loss=0.6533 val_mAP=0.6135 val_AUROC=0.6191
Epoch 08 | train_loss=0.6554 val_loss=0.6532 val_mAP=0.6136 val_AUROC=0.6191
Epoch 09 | train_loss=0.6552 val_loss=0.6533 val_mAP=0.6135 val_AUROC=0.6191
Epoch 10 | train_loss=0.6551 val_loss=0.6533 val_mAP=0.6135 val_AUROC=0.6191

Loaded best model with validation mAP: 0.6136


## 5) Prediction Utilities (Corrected for 8-bit)

In [11]:
def predict_top_k_od(input_word: str, input_bit: int, top_k: int = 10):
    """
    Return top-k OD (word, bit) pairs with highest predicted probability.
    """
    x = np.asarray([[le_word_joint.transform([str(input_word)])[0], int(input_bit)]], dtype=np.int64)
    x = scaler_joint.transform(x.astype(np.float32))
    
    with torch.no_grad():
        logits = model_joint(torch.from_numpy(x).float())
        probs = torch.sigmoid(logits).cpu().numpy()[0]  # shape (128,)
    
    # Decode indices -> (word, bit) for 8-bit words
    idxs = np.argsort(-probs)[:top_k]
    results = []
    for idx in idxs:
        od_word = idx // 8  # 8 bits per word
        od_bit = idx % 8    # bit position within word
        results.append((int(od_word), int(od_bit), float(probs[idx])))
    return results

In [12]:
# Example usage (prints top-5 OD pairs)
print("\nExample: top-5 OD pairs for ('c', 0)")
for w, b, p in predict_top_k_od('c', 0, top_k=5):
    print(f"  OD_word={w:2d}, OD_bit={b:2d}, prob={p:.4f}")


Example: top-5 OD pairs for ('c', 0)
  OD_word= 0, OD_bit= 1, prob=0.9668
  OD_word= 1, OD_bit= 7, prob=0.8778
  OD_word= 1, OD_bit= 2, prob=0.8690
  OD_word=10, OD_bit= 7, prob=0.8211
  OD_word= 2, OD_bit= 3, prob=0.8112


## 6) Model Evaluation

In [13]:
# Evaluate model on validation set with concise summaries
val_loss, val_ap, val_auc = evaluate(model_joint, val_loader)
print(f"Final Validation: loss={val_loss:.4f}, mAP={val_ap:.4f}, AUROC={val_auc:.4f}")

# Quick sanity check across a few inputs
examples = [('c', 0), ('d', 5), ('e', 7), ('f', 3)]
for iw, ib in examples:
    top = predict_top_k_od(iw, ib, top_k=5)
    print(f"\nInput ({iw}, {ib}) -> top-5 OD (word, bit, prob):")
    for w, b, p in top:
        print(f"  ({w:2d}, {b:2d}) p={p:.4f}")
        # Verify bit is in valid range
        assert 0 <= b <= 7, f"Invalid bit index {b}, should be 0-7"

Final Validation: loss=0.6532, mAP=0.6136, AUROC=0.6191

Input (c, 0) -> top-5 OD (word, bit, prob):
  ( 0,  1) p=0.9668
  ( 1,  7) p=0.8778
  ( 1,  2) p=0.8690
  (10,  7) p=0.8211
  ( 2,  3) p=0.8112

Input (d, 5) -> top-5 OD (word, bit, prob):
  ( 1,  4) p=0.9324
  (12,  5) p=0.8647
  ( 3,  6) p=0.8645
  ( 2,  2) p=0.8586
  (14,  4) p=0.8573

Input (e, 7) -> top-5 OD (word, bit, prob):
  ( 2,  2) p=0.9589
  (13,  0) p=0.8684
  ( 3,  3) p=0.8485
  (13,  3) p=0.8417
  (15,  2) p=0.7799

Input (f, 3) -> top-5 OD (word, bit, prob):
  ( 3,  6) p=0.9786
  ( 0,  7) p=0.9006
  (14,  4) p=0.8812
  (14,  7) p=0.8649
  ( 9,  7) p=0.7751


## 7) Save Model

In [14]:
# Save model and encoders for later reuse
save_bundle = {
    'state_dict': model_joint.state_dict(),
    'scaler': scaler_joint,
    'label_encoder_word': le_word_joint,
    'input_dim': X_train.shape[1],
    'hidden': 256,
    'output_dim': 128,  # 16 words × 8 bits
}

# Torch model is saved as state_dict alongside scikit objects
joblib.dump(save_bundle, 'joint_od_predictor_8bit_bundle.pkl')
print("Saved: joint_od_predictor_8bit_bundle.pkl")

print("\nUsage note:")
print("To load later:")
print("  bundle = joblib.load('joint_od_predictor_8bit_bundle.pkl')")
print("  model = ODJointMLP(in_dim=bundle['input_dim'], hidden=bundle['hidden'], out_dim=bundle['output_dim'])")
print("  model.load_state_dict(bundle['state_dict'])")
print("  scaler = bundle['scaler']")
print("  le_word = bundle['label_encoder_word']")

Saved: joint_od_predictor_8bit_bundle.pkl

Usage note:
To load later:
  bundle = joblib.load('joint_od_predictor_8bit_bundle.pkl')
  model = ODJointMLP(in_dim=bundle['input_dim'], hidden=bundle['hidden'], out_dim=bundle['output_dim'])
  model.load_state_dict(bundle['state_dict'])
  scaler = bundle['scaler']
  le_word = bundle['label_encoder_word']


## 8) Advanced Prediction Utilities (8-bit corrected)

In [15]:
def predict_od_matrix(input_word: str, input_bit: int) -> np.ndarray:
    """
    Return a (16, 8) matrix of P(OD_word=w, OD_bit=b | input) from DL model.
    """
    x = np.asarray([[le_word_joint.transform([str(input_word)])[0], int(input_bit)]], dtype=np.int64)
    x = scaler_joint.transform(x.astype(np.float32))
    
    with torch.no_grad():
        logits = model_joint(torch.from_numpy(x).float())
        probs = torch.sigmoid(logits).cpu().numpy()[0]  # shape (128,)
    
    return probs.reshape(16, 8)

def rank_output_words(prob_matrix: np.ndarray, agg: str = 'sum'):
    """
    Rank output words by aggregated bit probability.
    agg in {'sum','max','mean'} determines how to aggregate across 8 bits.
    Returns list of tuples (word, score).
    """
    if agg == 'sum':
        scores = prob_matrix.sum(axis=1)
    elif agg == 'max':
        scores = prob_matrix.max(axis=1)
    elif agg == 'mean':
        scores = prob_matrix.mean(axis=1)
    else:
        raise ValueError("agg must be one of {'sum','max','mean'}")
    
    ranking = sorted([(w, float(scores[w])) for w in range(16)], key=lambda x: x[1], reverse=True)
    return ranking

def rank_output_bits(prob_matrix: np.ndarray, output_word: int, top_k: int = 8):
    """
    Rank bits within a chosen output word. Returns top_k list of (bit, prob).
    """
    row = prob_matrix[output_word]
    idxs = np.argsort(-row)[:min(top_k, 8)]  # Max 8 bits per word
    return [(int(b), float(row[b])) for b in idxs]

def predict_next_round_dl(input_word: str, input_bit: int, top_k_words: int = 5, bit_top_k: int = 8, agg: str = 'sum'):
    """
    DL-only interface to get best output words and bits.
    Returns dict with:
      - prob_matrix: (16,8) numpy array
      - top_words: list[(word, score)] of length top_k_words
      - top_bits_per_word: dict[word] -> list[(bit, prob)] of length bit_top_k
    """
    pm = predict_od_matrix(input_word, input_bit)
    words_ranked = rank_output_words(pm, agg=agg)[:top_k_words]
    bits_per = {w: rank_output_bits(pm, w, top_k=min(bit_top_k, 8)) for w, _ in words_ranked}
    
    return {
        'prob_matrix': pm,
        'top_words': words_ranked,
        'top_bits_per_word': bits_per,
    }

In [28]:
# DL-only example usage
examples = [('c', 0), ('d', 5), ('e', 7)]
iw=['c','d','e']
ID_OD=[]
for w in iw:
    for b in range(8):
        res=predict_next_round_dl(w,b,top_k_words=1,bit_top_k=1,agg='sum');
        # print(f"\nInput ({w}, {b})")

        # print("Top output words (agg=sum):")
        # for cw, s in res['top_words']:
            # print(f"  word={cw:2d} score={s:.4f}")
        
        for cw, _ in res['top_words']:
            for ib, p in res['top_bits_per_word'][cw]:
                ID_OD.append([(w,b),(cw,ib),p])
                # print(f"    bit={ib:2d} p={p:.4f}")
                # Verify bit is in valid range
                assert 0 <= ib <= 7, f"Invalid bit index {ib}, should be 0-7"
for ID,OD,p in ID_OD:
    print(f"ID:{ID}",end=" ")
    print(f"OD:{OD}",end=" ")
    print(f"Probability:{p:.5f}",end="\n")
# for iw, ib in examples:
#     res = predict_next_round_dl(iw, ib, top_k_words=3, bit_top_k=5, agg='sum')
#     print(f"\nInput ({iw}, {ib})")
#     print("Top output words (agg=sum):")
#     for w, s in res['top_words']:
#         print(f"  word={w:2d} score={s:.4f}")
    
#     for w, _ in res['top_words']:
#         print(f"  Best bits in word {w}:")
#         for b, p in res['top_bits_per_word'][w]:
#             print(f"    bit={b:2d} p={p:.4f}")
#             # Verify bit is in valid range
#             assert 0 <= b <= 7, f"Invalid bit index {b}, should be 0-7"

ID:('c', 0) OD:(13, 1) Probability:0.80892
ID:('c', 1) OD:(13, 0) Probability:0.83634
ID:('c', 2) OD:(2, 1) Probability:0.74452
ID:('c', 3) OD:(5, 6) Probability:0.75702
ID:('c', 4) OD:(2, 7) Probability:0.84758
ID:('c', 5) OD:(2, 6) Probability:0.88548
ID:('c', 6) OD:(2, 5) Probability:0.78956
ID:('c', 7) OD:(13, 2) Probability:0.71341
ID:('d', 0) OD:(3, 3) Probability:0.82803
ID:('d', 1) OD:(14, 0) Probability:0.84023
ID:('d', 2) OD:(3, 1) Probability:0.75383
ID:('d', 3) OD:(6, 6) Probability:0.77219
ID:('d', 4) OD:(14, 5) Probability:0.87043
ID:('d', 5) OD:(14, 4) Probability:0.85729
ID:('d', 6) OD:(3, 5) Probability:0.79634
ID:('d', 7) OD:(3, 4) Probability:0.73760
ID:('e', 0) OD:(0, 3) Probability:0.82100
ID:('e', 1) OD:(0, 2) Probability:0.87288
ID:('e', 2) OD:(15, 7) Probability:0.74733
ID:('e', 3) OD:(7, 6) Probability:0.77664
ID:('e', 4) OD:(15, 5) Probability:0.84630
ID:('e', 5) OD:(15, 4) Probability:0.85209
ID:('e', 6) OD:(8, 1) Probability:0.68832
ID:('e', 7) OD:(15, 2) Pr